# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "myrna", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_myrna/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
# TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = spark.read.json(challenge_path)
challenge.printSchema()

root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)



In [0]:
bronze_cols = set(spark.table(f"{config.catalog}.{config.schema}.bronze_events").columns)
print("Campo nuevo:", set(challenge.columns) - bronze_cols)

Campo nuevo: {'app_version'}


In [0]:
bronze_types = {r.event_type for r in spark.table(f"{config.catalog}.{config.schema}.bronze_events")
                .select("event_type").distinct().collect()}
challenge_types = {r.event_type for r in challenge.select("event_type").distinct().collect()}
print("event_type nuevo:", challenge_types - bronze_types)

event_type nuevo: {'refund'}


In [0]:
print("Filas:", challenge.count())
print("IDs distintos:", challenge.select("event_id").distinct().count())

Filas: 10001
IDs distintos: 10000


## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

# TODO: unir por nombre admitiendo columnas faltantes, deduplicar y escribir target_table.

In [0]:
historic = (spark.table(source_table)
    .withColumn("_source", F.lit("bronze_events"))
    .withColumn("_source_file", F.lit(None).cast("string"))
    .withColumn("_ingested_at", F.current_timestamp()))
historic.show(5, truncate=False)

+-----------------------+-----------+--------+------------------------+-----------+----------+-------------+------------+--------------------------+
|context                |customer_id|event_id|event_ts                |event_type |product_id|_source      |_source_file|_ingested_at              |
+-----------------------+-----------+--------+------------------------+-----------+----------+-------------+------------+--------------------------+
|{android, session_1348}|2049       |0       |2026-03-04T11:36:41.000Z|search     |497       |bronze_events|NULL        |2026-09-26 15:21:59.276676|
|{android, session_9064}|3190       |1       |2026-03-04T08:12:05.000Z|checkout   |412       |bronze_events|NULL        |2026-09-26 15:21:59.276676|
|{android, session_3509}|2223       |2       |2026-03-03T04:44:14.000Z|add_to_cart|340       |bronze_events|NULL        |2026-09-26 15:21:59.276676|
|{web, session_4707}    |4951       |3       |2026-03-01T23:26:45.000Z|search     |292       |bronze_event

In [0]:
new_batch = (challenge
    .withColumn("_source", F.lit("events_json_v2"))
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp()))

new_batch.show(5, truncate=False)

+-----------+-----------------------+-----------+--------+------------------------+----------+----------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+
|app_version|context                |customer_id|event_id|event_ts                |event_type|product_id|_source       |_source_file                                                                                                                                        |_ingested_at              |
+-----------+-----------------------+-----------+--------+------------------------+----------+----------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+
|2.1.0      |{web, session_9543}    |3789       |205000  |2026-03-08T02:19:03.000Z|view      |222       |even

In [0]:
combined = historic.unionByName(new_batch, allowMissingColumns=True)
deduped = combined.dropDuplicates(["event_id"])

deduped.show(5, truncate=False)

+----------------------+-----------+--------+------------------------+-----------+----------+-------------+------------+--------------------------+-----------+
|context               |customer_id|event_id|event_ts                |event_type |product_id|_source      |_source_file|_ingested_at              |app_version|
+----------------------+-----------+--------+------------------------+-----------+----------+-------------+------------+--------------------------+-----------+
|{web, session_7918}   |4577       |28      |2026-03-04T06:33:37.000Z|search     |462       |bronze_events|NULL        |2026-09-26 15:26:20.231785|NULL       |
|{ios, session_1100}   |363        |29      |2026-03-06T22:50:22.000Z|search     |281       |bronze_events|NULL        |2026-09-26 15:26:20.231785|NULL       |
|{web, session_2399}   |2066       |30      |2026-03-05T01:01:59.000Z|view       |49        |bronze_events|NULL        |2026-09-26 15:26:20.231785|NULL       |
|{android, session_694}|1317       |33  

In [0]:
deduped.write.mode("overwrite").saveAsTable(target_table)

## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.table(target_table)
total_rows = result.count()
distinct_ids = result.select("event_id").distinct().count()
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"


print(f"OK: {total_rows} filas, {distinct_ids} ids distintos")

OK: 210000 filas, 210000 ids distintos


## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.